In [1]:
#requirements: numpy, pandas, openpyxl, SimpleITK, pydicom, platipy

import analyze_dcm as org
import analyze_ROI as ROI
import analyze_spacing as sp
import total as tot
import justified as just
import check_over as over
import check_rate as rate
import merge_df as merge_df
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
def main():
    # directory_dcm_files = (Path("//IHSR.dom/OSRFileServices") / "Ric.FisicaSanitaria" / "AAAshared" / "dataset" / 
    #                            "segmentazione" / "breast_Fodor22_upto2017" / "DB_Resume_Scripts" / "DB_uncomposed")
    
    directory_dcm_files = (Path.home() / "Desktop" / "Breast" / ".venv_Fin" / "ResumeScripts" / "DB" / "Prova")
    
    # directory_ct_files = (Path("//IHSR.dom/OSRFileServices") / "Ric.FisicaSanitaria" / "AAAshared" / "dataset" / 
    #                            "segmentazione" / "breast_Fodor22_upto2017" / "DB_Resume_Scripts" / "DB_composed")
    
    directory_ct_files = (Path.home() / "Desktop" / "Breast" / ".venv_Fin" / "ResumeScripts" / "DB" / "DB_composed")
    Path(directory_ct_files).mkdir(parents=True, exist_ok=True)
    
    directory_out = Path.home() / "Desktop" / "Breast" / ".venv_Fin" / "ResumeScripts" / "Analyses_Fin"
    Path(directory_out).mkdir(parents=True, exist_ok=True)

    divide = str(input("Do you need to reorganize files in images? (y/n)"))    
    
    if "y" in divide.lower():
        #if True split Dicom files
        organize_ct = org.divide_dcm(directory_dcm_files, directory_ct_files, divide=True)
        print("")
    else:
        organize_ct = org.divide_dcm(directory_dcm_files, directory_ct_files, divide=False)  
        print("")

    
    save_df = str(input("\nDo you need to save a dataframe with all headers info? (y/n)"))

    if "y" in save_df.lower():
        #if save=True create df with Dicom headers
        df_py = org.find_ct_info(directory_ct_files, directory_out, save_df=True)
        print("")
    else:
        df_py = org.find_ct_info(directory_ct_files, directory_out, save_df=False)
        print("")        

    
    save_ROI = str(input("\nDo you need to save a dataframe with all ROI's info? (y/n)"))
    
    if "y" in save_ROI.lower():
        #if True create db with all patient's ROI and relative counts
        df_ROI, df_counts = ROI.all_ROI(df_py, directory_out, save_ROI=True)          
        print("")
    else:
        df_ROI, df_counts = ROI.all_ROI(df_py, directory_out, save_ROI=False) 
        print("")


    save_sp = str(input("Do you need to save info about voxel spacing distribution? (y/n)"))
    
    if "y" in save_sp.lower():
        #if save=True create voxel spacing distribution for each dimension
        new_x, new_y, new_z = sp.read_spacing(df_py, directory_out, save_sp=True)
        print("")
    else:
        #if save=True create voxel spacing distribution for each dimension
        new_x, new_y, new_z = sp.read_spacing(df_py, directory_out, save_sp=False)
        print("")
    
    new_sp = np.array([new_x, new_y, new_z])
    print("The most common voxel spacing is: ", new_sp)
    print("")

    ID_problems = ["70230254", "70366136", "433906"] #bilaterali

    all = str(input("Do you want to analyze the histograms relating to the entire ROI under consideration? (y/n)"))
    
    if "y" in all.lower():
        all = True
        print("Analyzing total histograms is set on: ", all)
        print("")
        
        save_all = str(input("Do you want to save files and plots? (y/n)"))
        
        if "y" in save_all.lower():
            dir_files_fin = tot.res_and_create_histo(df_py, ID_problems, new_sp, directory_out, save_all = True)
            print("")
        else:
            dir_files_fin = tot.res_and_create_histo(df_py, ID_problems, new_sp, directory_out, save_all = False)
            print("")
    
    else:
        print("You preferred to not analyze the histograms.")
        print("")

        dir_files_fin = Path(directory_out) / "Total_ROI" / "Files_ok"
    
    just_or_not = str(input("Do you want to analyze a specific region? (y/n)"))

    if "y" in just_or_not.lower():
        just_or_not = True
        print("Analyzing histograms about specific region is set on: ", just_or_not)
        print("")
        
        save_just = str(input("Do you want to save files and plots? (y/n)"))        
        if "y" in save_just.lower():
            just.histo_just(dir_files_fin, directory_out, save_just=True)
            print("")
        else:
            just.histo_just(dir_files_fin, directory_out, save_just=False) 
            print("")
            
    else:
        print("You preferred to not analyze the histograms about specific region.")
        print("")

    check_over = str(input("Do you want to exclude patients with counts above an HU threshold? (y/n)"))
    
    if "y" in check_over.lower():
        check_over = True
        print("Analyzing patients searching significative above specific region is set on: ", check_over)
        print("")
        
        save_over = str(input("Do you want to save files and plots? (y/n)"))
        if "y" in save_over.lower(): 
            over.check_over(dir_files_fin, directory_out, save_over=True)
            print("")
        else:
            over.check_over(dir_files_fin, directory_out, save_over=False)
            print("")
            
    else:
        print("You preferred not to specifically analyze the histograms above the threshold.")
        print("")


    check_rate = str(input("Do you want to exclude patients with a significant area outside the HU thresholds? (y/n)"))
    if "y" in check_rate.lower():
        check_rate = True
        print("Analyzing patients searching significative outside specific region is set on: ", check_rate)
        print("")
        
        save_rate = str(input("Do you want to save files and plots? (y/n)"))
        if "s" in save_rate.lower():      
            rate.check_rate(dir_files_fin, directory_out, save_rate=True)
            print("")
        else:
            rate.check_rate(dir_files_fin, directory_out, save_rate=False)
            print("")
            
    else:
        print("You preferred not to specifically analyze the histograms outside the HU thresholds.")
        print("")


    if "y" in str(input("Do you want to merge clinical and densitometric database? (y/n)")).lower():
        print("You chose to merge databases.")

        show_merge = str(input("\nDo you want to see all the dataframes until the final one? (y/n)"))
        save_merge = str(input("Do you want to save the final database? (y/n)"))
        merge_df.merge_db(directory_out, show_merge, save_merge)
    
    else:
        print("You chose not to merge databases.")
        print("")

In [3]:
if __name__ == "__main__":
    main()

Do you need to reorganize files in images? (y/n) y


The number of different IDs are:3.
All patients have only one RTstruct files.
All files are inside 'ID', which contains CT_i, RTst and Altro
CT_i corresponds to different UID instances corresponding to RS_i inside RTst folder.




Do you need to save a dataframe with all headers info? (y/n) y


Save df headers DICOM is set on:  True

Dataframe with Dicom header saved in C:\Users\belardo.alfonso\Desktop\Breast\.venv_Fin\ResumeScripts\Analyses_Fin_PP\py_patient_file.xlsx




Do you need to save a dataframe with all ROI's info? (y/n) y


Save df ROI is set on:  True



IndexError: list index out of range